# 5.1 Boundary-based segmentation. Hough transform

Image segmentation is the process of **partitioning an image into regions by assigning a label to every pixel**, such that pixels with the same label share certain characteristics. Formally, it maps an image $I(x,y)$ to a **label map** $M(x,y)$, where $M(x,y)=k$ means that the pixel $(x,y)$ is assigned the label $k \in \{1,\dots,K\}$. The techniques addressing those problems can be classified into two major groups:

- Those producing regions with **similar appearance** or visual properties (*e.g.*, intensity, color, texture, or spatial proximity), addressed by **classical segmentation** methods.
- Those segmenting **meaningful entities** (*e.g.*, **semantic segmentation** where every pixel is assigned a semantic class label, or **instance segmentation** where each countable object instance gets its own mask, see Fig.1). This is faced by **learning-based segmentation** methods (deep neural networks).

<figure style="text-align:center">
  <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/image_segmentation/images/image-segmentation-example.png" width="550">
  <figcaption>Fig. 1. Example of instance segmentation where each region corresponds to an object in the scene.</figcaption>
</figure>$\\[5pt]$

Classical methods can be further classified into:
- **Boundary-based techniques**, which identify the image regions through the boundaries that separate them. Edge pixels enclose a region with similar intensities.
- **Threshold-based techniques**, which separate regions according to their pixel intensity.
- **Region-based techniques**, which form regions by grouping pixels according to their similarity in features and/or spatial relationships.
- **Graph-based techniques**, which represent the image as a weighted graph and partition it by cutting it.

Fig. 2 summarizes these classical techniques, together with the main types of learning-based segmentation.

<figure style="text-align:center">
  <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/image_segmentation/images/segmentation-techniques.png" width="800">
  <figcaption>Fig. 2. Summary of image segmentation techniques.</figcaption>
</figure>$\\[5pt]$


In this chapter we are going to experience boundary-based and region-based techniques, starting with **boundary-based techniques**, which are based on detecting specific boundaries in the image (e.g. circles). In this context, boundaries are defined as edge pixels that enclose a region.  

Boundary-based techniques could be roughly classified into:
- **Local techniques.** Try to segment regions by detecting closed boundaries, which typically enclose pixels with similar intensities.
   - LoG + zero crossing.
   - Edge following (Canny operator).
- **Global techniques.** Detect particular shapes in the image (circles, lines, etc.).
   - Hough transform. 
   
This notebook will cover the **Hough transform** (<a href="#511">section 5.1.1</a>), a boundary-based technique that can be used for detecting regions with an arbitrary shape in images, and its faster variant, the **probabilistic Hough transform** (<a href="#512">section 5.1.2</a>).

## Problem context - Self-driving car 

A prestigious company located at PTA (The Andalusia Technology Park) is organizing a [hackathon](https://en.wikipedia.org/wiki/Hackathon) for this year in order to motivate college students to make further progress in the autonomous cars field. Computer vision students at UMA decided to take part in it, but the organizers have posed an initial basic task to guarantee that participants have expertise in computer vision techniques. 

This way, the company sent to students a task for **implementing a basic detector of road lane lines using OpenCV in Python**. We are lucky! These are two tools that we know well ;).

Detecting road lane lines is a fundamental task for autonomous vehicles while driving on the road. It is the building block to other path planning and control actions like braking and steering. 

So here we are! We are going to detect road lane lines using the Hough transform in OpenCV.

<figure style="text-align:center">
  <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/image_segmentation/images/car.gif" width="400">
  <figcaption>Fig. 3. Example of road lane line detection from a car camera.</figcaption>
</figure>$\\[5pt]$

> 🔍 <font color="#0072B2"><b>AI Exploration:</b> Classical vs learning-based lane detection</font>  
> **Suggested prompt**  
> *In the context of self-driving cars, how do classical road lane line detectors (edge detection + Hough transform) and learning-based detectors compare? What are the advantages and drawbacks of each one?*

In [ ]:
# In case of using Google Colab, mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    # If your notebook and images are in your Drive, change directory accordingly:
    # %cd '/content/drive/MyDrive/Colab Notebooks/uma_cv_2027/Chapter 05. Image segmentation'
except ImportError:
    pass

import numpy as np
import cv2
import math
import matplotlib.pyplot as plt
import matplotlib
import scipy.stats as stats

matplotlib.rcParams['figure.figsize'] = (10.0, 10.0)
images_path = './images/'

## 5.1.1 Hough transform <a id="511"></a>

The **Hough Transform** is a technique for the detection of arbitrary shapes in an image. For that, such shapes must be expressed in:

- analytical form (**classic Hough**), e.g. using the mathematical representation of lines, circles, ellipses, etc., or 
- in a numerical form (**generalized Hough**) where the shape is given by a table. 

Since our goal is to detect lines, we will focus here on analytically expressed shapes. Specifically, a line can be represented analytically as:

$$y = ax + b$$

where $a$ is the slope and $b$ is the $y$-intercept, what is called the **explicit representation** of a line. However, this representation is problematic when representing vertical lines ($a=\infty$), and the possible values of $a$ cannot be discretized in a bounded range. Another option is its **normal form**:

$$\rho = x \cos \theta + y \sin \theta$$

where $\rho$ is the signed perpendicular distance from the origin $(0,0)$ to the line, and $\theta$ is the angle between the $x$ axis and this perpendicular line, measured clockwise since in images the $y$ axis points down (see Fig. 4). Thereby, we are going to represent lines using the pair of parameters $(\rho, \theta)$.

<figure style="text-align:center">
  <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/image_segmentation/images/houghlines.svg" width="250">
  <figcaption>Fig. 4. Normal form of a line.</figcaption>
</figure>$\\[5pt]$


The Hough transform works by a voting procedure, which is carried out in a parameter space ($\rho,\theta$ in our case). This technique consists of the following steps:

1. **Build an accumulator matrix**, where rows index the possible values of $\rho$, and columns those for $\theta$. For example, if the possible values for $\rho$ are $-d, \ldots, -1, 0, 1, \ldots, d$ (where $d$ is the max distance, e.g. the diagonal size of the image) and those for $\theta$ are $0^\circ, 1^\circ, \ldots, 179^\circ$, the matrix shape would be $(2d+1, 180)$.

In [ ]:
# Define possible rho and theta values
theta_values = [0,np.pi/4,np.pi/2,3*np.pi/4]
d = 5 # Max distance
rho_values = list(range(-d, d+1)) # rho is signed: from -d to d

# Create the accumulator
acc = np.zeros([len(rho_values),len(theta_values)])

2. **Obtain the edge points** that are candidates to belong to the shape boundaries: apply an edge detector and binarize its response (e.g. by thresholding the gradient magnitude). Some detectors, like Canny, already return a binary edge image.

In [ ]:
# Coordinates of white points in binary image
xs = np.array([1,2,3])
ys = np.array([4,3,2])

# Initial image
plt.subplot(221)
blank_image = np.zeros((6,4,1), np.uint8)
blank_image[ys,xs] = 255
plt.imshow(blank_image, cmap='gray');
plt.title('Initial binary image')

# Show them!
plt.subplot(222)
plt.plot(xs,ys,'rx')
plt.axis('scaled')    
axes = plt.gca()
axes.set_xlim([0,4])
axes.set_ylim([0,6])
plt.gca().invert_yaxis()
plt.title('White pixels in binary image');


3. For each candidate (white pixel, that is, **edge point**):
   1. **Evaluate**: Since the point coordinates $(x,y)$ are known, place them in the normal form of the line and iterate over the possible values of $\theta$ to obtain the values for $\rho$. In the previous example, $\rho_i = x \cos \theta_i + y \sin \theta_i,\ i \in \{0, \ldots, 179\}$.
   2. **Vote**: For every obtained pair $(\rho_i, \theta_i)$, round $\rho_i$ to the closest value of $\rho$ in the accumulator (e.g. $3.54 \rightarrow 4$) and increment by one the value of its associated cell.

If we do this with an accumulator with enough resolution, we would get a sinusoidal curve $\rho(\theta) = x \cos \theta + y \sin \theta$ in the parameter space.

In [ ]:
# For each white pixel

for i in range (0,len(xs)):
    x = xs[i]
    y = ys[i]
    
    # Show the point voting
    subplot_index = str(32) + str(i*2+1)
    plt.subplot(int(subplot_index))   
    plt.axis('scaled')    
    axes = plt.gca()
    axes.set_xlim([0,4])
    axes.set_ylim([0,6])
  
    plt.plot(xs,ys,'rx')
    plt.plot(x,y,'co')
    
    plt.title('Pixel being evaluated at iteration ' + str(i+1))
    plt.gca().invert_yaxis()
    
    
    # Evaluate the (x,y) coordinates for different values of theta, and 
    # retrieve rho
    for theta_index in range(0,len(theta_values)):
        theta = theta_values[theta_index]
        rho = x*np.cos(theta) + y*np.sin(theta)
        rho = int(np.round(rho))
        # Vote! (rows go from rho = -d to rho = d, so rho is stored in row rho + d)
        acc[rho + d][theta_index] += 1.0
        
    # Show the accumulator
    subplot_index = str(32) + str(i*2+2)
    plt.subplot(int(subplot_index))        
    plt.imshow(acc,cmap='gray',vmax=3,aspect='auto');    
    plt.xticks([0, 1, 2, 3], [r'$0$', r'$\pi/4$', r'$\pi/2$', r'$3\pi/4$'])
    plt.yticks(range(len(rho_values)), rho_values)
                                                       
    plt.title('Accumulator values')

4. Finally, **obtain the shape candidates** by setting a threshold to control how many votes a pair $(\rho, \theta)$ needs to be considered a line, and by looking for local maxima (**peaks**) in the accumulator.

Fig. 5 shows the whole procedure: each edge point (left) votes for all the lines that go through it, drawing a sinusoidal curve in the accumulator (right). The peak gives the detected line.

<figure style="text-align:center">
  <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/image_segmentation/images/houghlinesdemo_v2.gif" width="800" align="center">
  <figcaption>Fig. 5. Left, image space. Right, parameter space illustrating the evolution of the votes. Note that in this example $\theta$ has only 12 possible values ($0^\circ, 15^\circ, \ldots, 165^\circ$).</figcaption>
</figure>$\\[5pt]$


The idea behind this algorithm is that each pixel votes for all the lines that go through it. Then, when a second pixel votes, the line connecting both pixels gets two votes, since it goes through both of them. The more pixels a line goes through, the more votes it gets, which is why the lines of the image appear as peaks in the accumulator. 

> 🛠️ <font color="#0B6B3A"><b>AI Coding Support:</b> Detecting lines with OpenCV</font>  
> In practice, the Hough transform for detecting lines is already implemented in the OpenCV function [`cv2.HoughLines()`](https://docs.opencv.org/4.2.0/dd/d1a/group__imgproc__feature.html#ga46b4e588934f6c8dfd509cc6e0e4545a), so let's see how it works!

> **Suggested prompt**  
> *I need to use OpenCV to detect lines in an image with the Hough transform through the `cv2.HoughLines()` function. Do not write the full solution. Explain how its main parameters (`image`, `rho`, `theta` and `threshold`) relate to the accumulator and the voting procedure of the Hough transform, which values are typically used, and what output I should expect.*

Prior to using `cv2.HoughLines()`, and as commented in step 2 of the algorithm, a binary image is needed. For that we are going to resort to our old friend the Canny algorithm, so the detected edges will be the white pixels in the binary image.

> 🛠️ <font color="#0B6B3A"><b>AI Coding Support:</b> Remembering Canny in OpenCV</font>  
> **Suggested prompt**  
> *Briefly remind me what the OpenCV function `cv2.Canny()` expects as input (image type and main parameters) and what it returns. Keep it short and do not write the full solution.*

As we know, noisy images seriously hamper the performance of computer vision techniques, and since [cv2.Canny()](https://docs.opencv.org/4.2.0/dd/d1a/group__imgproc__feature.html#ga04723e007ed888ddf11d9ba04e2232de) does not include blurring, we provide here a method called `gaussian_smoothing()` to assist you in that task.

In [ ]:
def gaussian_smoothing(image, sigma, w_kernel):
    """ Blur and normalize input image.   
    
        Args:
            image: Input image to be binarized
            sigma: Standard deviation of the Gaussian distribution
            w_kernel: Kernel aperture size
                    
        Returns: 
            binarized: Blurred image
    """   
    
    # Define 1D kernel
    s=sigma
    w=w_kernel
    kernel_1D = [np.exp(-z*z/(2*s*s))/np.sqrt(2*np.pi*s*s) for z in range(-w,w+1)]
    
    # Apply distributive property of convolution
    kernel_2D = np.outer(kernel_1D,kernel_1D)
    
    # Blur image
    smoothed_img = cv2.filter2D(image,cv2.CV_8U,kernel_2D)
    
    # Normalize to [0 255] values
    smoothed_norm = np.array(image.shape)
    smoothed_norm = cv2.normalize(smoothed_img,None, 0, 255, cv2.NORM_MINMAX)
    
    return smoothed_norm

### **<span style="color:green"><b><i>ASSIGNMENT 1: Detecting lines with Hough</i></b></span>**

**Your first task is** to apply [`cv2.HoughLines()`](https://docs.opencv.org/4.2.0/dd/d1a/group__imgproc__feature.html#ga46b4e588934f6c8dfd509cc6e0e4545a) to the image `car.png`, a test image taken from the frontal camera of a car. Draw the resultant lines using [`cv2.line()`](https://docs.opencv.org/4.2.0/d6/d6e/group__imgproc__draw.html#ga7078a9fae8c7e7d13d24dac2520ae4a2).

*Note that `cv2.HoughLines()` returns a $( n_\text{lines} \times 1 \times 2 )$ array containing, in each row, the $(\rho, \theta)$ pair of each detected line, where $\theta \in [0, \pi)$ and $\rho$ can be negative. For drawing the lines, you have to [transform the resultant lines](https://docs.opencv.org/4.x/d9/db0/tutorial_hough_lines.html) from the $(\rho, \theta)$ space to Cartesian coordinates.*

Try different parameter values until you get something like this:

<figure style="text-align:center">
  <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/image_segmentation/images/hough-lines-result.png" align="center">
  <figcaption>Fig. 6. Example of lines detection.</figcaption>
</figure>$\\[5pt]$


In [ ]:
# Assignment 1
# Read the image
image = cv2.imread(images_path + "car.png",-1)

# Convert to RGB and get gray image
image = cv2.cvtColor(image, None)
gray = cv2.cvtColor(image, None)

# Blur the gray image
gray = gaussian_smoothing(None,2,5)

# Apply Canny algorithm
edges = cv2.Canny(None,100,200,apertureSize = 3)

# Search for lines using Hough transform
lines = cv2.HoughLines(None, rho=None, theta=None, threshold=None)

# For each line
for i in range(0, len(lines)):
    
    # Transform from polar coordinates to Cartesian coordinates
    rho = lines[i][0][None]
    theta = lines[i][0][None]
    
    a = math.cos(None)
    b = math.sin(None)
    
    x0 = None * None
    y0 = None * None
    
    # Get two points in that line 
    x1 = int(x0 + 2000*(-b));
    y1 = int(y0 + 2000*(a))
    pt1 = (x1,y1)
    x2 = int(x0 - 2000*(-b))
    y2 = int(y0 - 2000*(a))
    pt2 = (x2,y2)
    
    # Draw the line in the RGB image
    cv2.line(image, None, None, (255,0,0), 3, cv2.LINE_AA)

# Show resultant image
plt.imshow(image);

## 5.1.2 Probabilistic Hough transform <a id="512"></a>

For high-resolution images and large accumulator sizes the Hough transform may need long execution times. However, in applications like autonomous cars a fast execution is mandatory. For example, having a car moving at 100 km/h covers $\sim$ 28 meters in a second. Imagine how much the lines can change in that time!

The **probabilistic Hough Transform** is a more complex implementation of the Hough Line Transform. This alternative does not take all the points in the binary image into account, but a random subset of them that are still enough for line detection. This also results in lower thresholds when deciding if a line exists or not.

In its *progressive* version, random edge points vote one at a time. As soon as a cell of the accumulator exceeds the threshold, a line has been found: the edge points along it are joined into a **segment**, and they are removed from the image together with their votes. This way, many edge points never need to vote.

Fig. 7 shows this procedure: random edge points (left) vote in the accumulator (right) until a cell exceeds the threshold, and then the detected segment is drawn and its points are removed.

<figure style="text-align:center">
  <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/image_segmentation/images/probabilistic_houghlinesdemo.gif" width="800" align="center">
  <figcaption>Fig. 7. Progressive probabilistic Hough transform: random edge points vote until a line is found, and the points of the detected segment are removed.</figcaption>
</figure>$\\[5pt]$

> 🛠️ <font color="#0B6B3A"><b>AI Coding Support:</b> Probabilistic Hough transform in OpenCV</font>  
> OpenCV implements the probabilistic Hough transform in the function [`cv2.HoughLinesP()`](https://docs.opencv.org/4.2.0/dd/d1a/group__imgproc__feature.html#ga8618180a5948286384e3b7ca02f6feeb), so let's see how it works!

> **Suggested prompt**  
> *I need to use OpenCV to detect line segments with the probabilistic Hough transform through the `cv2.HoughLinesP()` function. Do not write the full solution. Explain what its main parameters mean, including `minLineLength` and `maxLineGap`, how they differ from those of `cv2.HoughLines()`, and what output I should expect.*

### **<span style="color:green"><b><i>ASSIGNMENT 2: Another option for detecting lines</i></b></span>**

Apply [`cv2.HoughLinesP()`](https://docs.opencv.org/4.2.0/dd/d1a/group__imgproc__feature.html#ga8618180a5948286384e3b7ca02f6feeb) to the image `car.png` and draw the detected segments. Try different parameter values until you get something like this:

<figure style="text-align:center">
  <img src="https://raw.githubusercontent.com/jotaraul/cv_hub/refs/heads/main/image_segmentation/images/probabilistic-hough-lines-result.png" align="center">
  <figcaption>Fig. 8. Lines detection example with the probabilistic Hough Transform.</figcaption>
</figure>$\\[5pt]$



In [ ]:
# Assignment 2
# Read the image
image = cv2.imread(images_path + "car.png",-1)

# Convert to RGB and get gray image
image = cv2.cvtColor(image, None)
gray = cv2.cvtColor(image, None)

# Blur the gray image
gray = gaussian_smoothing(None,1,2)

# Apply Canny algorithm
edges = cv2.Canny(None,100,200,apertureSize = 3)

# Search for lines using probabilistic Hough transform
rho = None
theta = None
threshold = None
lines = cv2.HoughLinesP(edges, rho, theta, threshold,
                        minLineLength=None,maxLineGap=None)
# For each line
for line in lines:
    
    # Draw the line in the RGB image
    x1,y1,x2,y2 = None
    cv2.line(image,(x1,y1),(x2,y2),(255,0,0),2)

# Show resultant image
plt.imshow(image);

> 🧭 <font color="#B8860B"><b>AI Reflection:</b> Lines vs segments</font>  
> **Suggested prompt**  
> *I have detected lines in the same image with `cv2.HoughLines()` and `cv2.HoughLinesP()`. Help me compare both results: ask me about the type of output (infinite lines vs segments), spurious detections and the effect of `minLineLength` and `maxLineGap`. Do not give me the conclusions; guide me so that I can write them myself.*

✍️ **Your notes / key takeaways:**

<p style="margin: 4px 0px 6px 5px; color:blue"><i>Write your conclusions here after discussing with your assistant!</i></p>

### <font color="blue"><b><i>Thinking about it (1)</i></b></font>

Now that you have played with the Hough Transform, **answer the following questions**:

- In the first assignment, we obtained an image with a number of red lines drawn on it. However, these lines go over pixels in the image that do not contain lines! (*e.g.* belonging to the sky). What could we do to fix this, that is, obtain the pixels belonging to lines in the image?
  
    <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>
    
- Without restrictions regarding execution time, which method would you use, Hough Transform or its probabilistic counterpart?
  
    <p style="margin: 4px 0px 0px 5px; color:blue"><i>Your answer here!</i></p> 
    
- Could there be a cell in the accumulator with a value higher than the number of white pixels in the binary image? why?
  
    <p style="margin: 4px 0px 0px 5px; color:blue"><i>Your answer here!</i></p> 
    
- Which should be the maximum value for $\rho$ in the accumulator? Why?
  
    <p style="margin: 4px 0px 0px 5px; color:blue"><i>Your answer here!</i></p>     

### **<span style="color:green"><b><i>OPTIONAL</i></b></span>**

Think about any other application where the finding of straight lines could be useful. Get some images related to said application and detect lines with the Hough transform!

### **<span style="color:green"><b><i>END OF OPTIONAL PART</i></b></span>**

## Conclusion

Terrific work! In the context of detecting road lane lines, you have learned:
- to detect shapes in images using the Hough transform.

Also, you obtained some knowledge about: 

- self-driving cars and computer vision, and
- lane line detection for autonomous cars.

See you in the next one! Keep learning!


## <font color="orange"><b><i>AI Appendix</i></b></font>

- **Model/tool used:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Your answer here!</i></p>

- **What I used AI for:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Briefly explain how you used AI in this notebook: whether you followed the proposed AI interactions, adapted them, skipped some of them, or used AI for additional purposes such as debugging, clarifying concepts, improving prompts or checking your reasoning.</i></p>

- **Most and least useful proposed AI interactions:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>By proposed AI interactions, we mean the AI prompts explicitly included in the notebook. Choose one that helped you the most and one that helped you the least. Briefly explain why.</i></p>

- **Most useful personal AI interaction:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>By personal AI interaction, we mean any additional prompt or follow-up question that you wrote yourself, beyond the prompts proposed in the notebook. Paste or summarize one, if any, and explain why it helped.</i></p>

- **What I verified, adapted or decided myself:**
  <p style="margin: 4px 0px 6px 5px; color:blue"><i>Briefly explain what you checked, corrected, adapted or decided before using the AI output in your final work.</i></p>